# Install mlxtend
Install the mlxtend library, which provides the Apriori algorithm.

In [10]:
!conda install -c conda-forge mlxtend -y

Jupyter detected...
3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

# All requested packages already installed.





==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c defaults conda




# Verify mlxtend Version

In [18]:
import mlxtend
print(mlxtend.__version__)

0.24.0


# Load Dataset
Load the Megastore dataset and inspect the first few rows.

In [19]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_csv("Megastore Dataset.csv")
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (8234, 14)


,OrderID,ProductName,Quantity,InvoiceDate,UnitPrice,TotalCost,Country,DiscountApplied,OrderPriority,Region,Segment,ExpeditedShipping,PaymentMethod,CustomerOrderSatisfaction
0,536370,INFLATABLE POLITICAL GLOBE,48,12/1/2010 8:45,$0.85,$40.80,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
1,536370,SET2 RED RETROSPOT TEA TOWELS,18,12/1/2010 8:45,$2.95,$53.10,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
2,536370,PANDA AND BUNNIES STICKER SHEET,12,12/1/2010 8:45,$0.85,$10.20,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
3,536370,RED TOADSTOOL LED NIGHT LIGHT,24,12/1/2010 8:45,$1.65,$39.60,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
4,536370,VINTAGE HEADS AND TAILS CARD GAME,24,12/1/2010 8:45,$1.25,$30.00,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied


# Data Cleaning

In [20]:
# FIX: was 'df_cleaned = df_cleaned.dropna()' — df_cleaned was never defined (NameError)
# FIX: cleaning moved before encoding so df is clean for all downstream steps
df = df.dropna()
df = df.drop_duplicates()
print(f"Shape after cleaning: {df.shape}")
print(df.head())

Shape after cleaning: (8218, 14)
   OrderID                         ProductName  Quantity     InvoiceDate  \
0   536370         INFLATABLE POLITICAL GLOBE         48  12/1/2010 8:45   
1   536370      SET2 RED RETROSPOT TEA TOWELS         18  12/1/2010 8:45   
2   536370     PANDA AND BUNNIES STICKER SHEET        12  12/1/2010 8:45   
3   536370       RED TOADSTOOL LED NIGHT LIGHT        24  12/1/2010 8:45   
4   536370  VINTAGE HEADS AND TAILS CARD GAME         24  12/1/2010 8:45   

  UnitPrice  TotalCost         Country DiscountApplied OrderPriority  \
0     $0.85     $40.80   United States             Yes          High   
1     $2.95     $53.10   United States             Yes          High   
2     $0.85     $10.20   United States             Yes          High   
3     $1.65     $39.60   United States             Yes          High   
4     $1.25     $30.00   United States             Yes          High   

      Region    Segment ExpeditedShipping PaymentMethod  \
0  Northeast  Corp

#  Encode Categorical Variables


In [21]:
#  Ordinal Encoding 
# OrderPriority: Medium=1, High=2 (preserves rank)
df['OrderPriority_Encoded'] = df['OrderPriority'].map({'Medium': 1, 'High': 2})

# CustomerOrderSatisfaction: 5-level scale (0=no response → 4=Very Satisfied)
customer_satisfaction_mapping = {
    'Prefer to not respond': 0,
    'Dissatisfied': 1,
    'Very dissatisfied': 2,
    'Satisfied': 3,
    'Very Satisfied': 4
}
df['CustomerOrderSatisfaction_Encoded'] = df['CustomerOrderSatisfaction'].map(customer_satisfaction_mapping)
#  One-Hot Encoding 
# FIX: ordinal columns now exist on df before get_dummies, so they carry through
# automatically — no redundant re-assignment needed after get_dummies
df_encoded = pd.get_dummies(df, columns=['Segment', 'PaymentMethod'])

# Confirm all encoded columns are present
print(df_encoded[['OrderPriority_Encoded', 'CustomerOrderSatisfaction_Encoded',
                   'Segment_Consumer', 'Segment_Corporate',
                   'PaymentMethod_Credit Card', 'PaymentMethod_PayPal']].head())

   OrderPriority_Encoded  CustomerOrderSatisfaction_Encoded  Segment_Consumer  \
0                      2                                3.0             False   
1                      2                                3.0             False   
2                      2                                3.0             False   
3                      2                                3.0             False   
4                      2                                3.0             False   

   Segment_Corporate  PaymentMethod_Credit Card  PaymentMethod_PayPal  
0               True                       True                 False  
1               True                       True                 False  
2               True                       True                 False  
3               True                       True                 False  
4               True                       True                 False  


# Export Encoded Dataset

In [22]:
# FIX: export of df_encoded was missing entirely — required by Task C1d
df_encoded.to_csv("encoded_dataset.csv", index=False)
print("Encoded dataset exported to encoded_dataset.csv")
print(f"Shape: {df_encoded.shape}")
print(f"Columns: {list(df_encoded.columns)}")

Encoded dataset exported to encoded_dataset.csv
Shape: (8218, 18)
Columns: ['OrderID', 'ProductName', 'Quantity', 'InvoiceDate', 'UnitPrice ', 'TotalCost ', 'Country', 'DiscountApplied', 'OrderPriority', 'Region', 'ExpeditedShipping', 'CustomerOrderSatisfaction', 'OrderPriority_Encoded', 'CustomerOrderSatisfaction_Encoded', 'Segment_Consumer', 'Segment_Corporate', 'PaymentMethod_Credit Card', 'PaymentMethod_PayPal']


## C2a:Transactionalize the Dataset

In [23]:
# Convert the dataset into a binary transaction matrix for Apriori.
# FIX: removed 'df = pd.read_csv(...)' re-load that was overwriting the cleaned df
# The basket is now built from the already-cleaned df (from Cell 4)

# Transactionalize: each row = one order, each column = one product, value = True/False
basket = df.groupby(['OrderID', 'ProductName'])['Quantity'].sum().unstack().fillna(0)

# FIX: index=True preserves OrderID (the index) in the exported CSV
# Original code used index=False which silently dropped OrderID
basket.to_csv("transactionalized_dataset.csv", index=True)
print("Transactionalized dataset exported to transactionalized_dataset.csv")

# Convert to boolean for Apriori
basket = basket.gt(0).astype(bool)
print(f"Basket shape: {basket.shape}")
print(basket.head())

Transactionalized dataset exported to transactionalized_dataset.csv
Basket shape: (441, 1562)
ProductName   50S CHRISTMAS GIFT BAG LARGE   DOLLY GIRL BEAKER  \
OrderID                                                          
536370                               False               False   
536852                               False               False   
536974                               False               False   
537065                               False               False   
537463                               False               False   

ProductName   I LOVE LONDON MINI BACKPACK   NINE DRAWER OFFICE TIDY  \
OrderID                                                               
536370                              False                     False   
536852                              False                     False   
536974                              False                     False   
537065                              False                     False   
537463           

## C2b: Export Merged Encoded and Transactionalized Dataset (Final Cleaned Dataset)

In [24]:
# Select only the encoded metadata columns
df_encoded = df_encoded[[
    'OrderID',
    'OrderPriority_Encoded',
    'CustomerOrderSatisfaction_Encoded',
    'Segment_Consumer',
    'Segment_Corporate',
    'PaymentMethod_Credit Card',
    'PaymentMethod_PayPal'
]]

# Merge encoded metadata with the transactional basket
df_cleaned_final = pd.merge(
    df_encoded,
    basket,
    left_on='OrderID',
    right_index=True,
    how='inner'   # FIX: 'inner' ensures only orders present in both are included
)

print("First few rows of merged dataset:")
print(df_cleaned_final.head())
print(f"\nShape: {df_cleaned_final.shape}")
print(df_cleaned_final.info())

df_cleaned_final.to_csv("cleaned_data_for_apriori.csv", index=False)
print("\nMerged encoded dataset saved to cleaned_data_for_apriori.csv")

First few rows of merged dataset:
   OrderID  OrderPriority_Encoded  CustomerOrderSatisfaction_Encoded  \
0   536370                      2                                3.0   
1   536370                      2                                3.0   
2   536370                      2                                3.0   
3   536370                      2                                3.0   
4   536370                      2                                3.0   

   Segment_Consumer  Segment_Corporate  PaymentMethod_Credit Card  \
0             False               True                       True   
1             False               True                       True   
2             False               True                       True   
3             False               True                       True   
4             False               True                       True   

   PaymentMethod_PayPal   50S CHRISTMAS GIFT BAG LARGE   DOLLY GIRL BEAKER  \
0                 False                 

## C2c: Run Apriori Algorithm

In [25]:
# Apply Apriori algorithm to the boolean basket
frequent_itemsets = apriori(basket, min_support=0.01, use_colnames=True)
print(f"Number of frequent itemsets found: {len(frequent_itemsets)}")
print(frequent_itemsets.head(10))

Number of frequent itemsets found: 8340
    support                               itemsets
0  0.020408                   ( DOLLY GIRL BEAKER)
1  0.011338         ( I LOVE LONDON MINI BACKPACK)
2  0.013605     ( SET 2 TEA TOWELS I LOVE LONDON )
3  0.036281              ( SPACEBOY BABY GIFT SET)
4  0.027211               (10 COLOUR SPACEBOY PEN)
5  0.013605           (12 COLOURED PARTY BALLOONS)
6  0.013605        (12 PENCIL SMALL TUBE WOODLAND)
7  0.015873  (12 PENCILS SMALL TUBE RED RETROSPOT)
8  0.015873          (12 PENCILS SMALL TUBE SKULL)
9  0.022676   (12 PENCILS TALL TUBE RED RETROSPOT)


## C2c: Generate Association Rules

In [26]:
# FIX: added num_itemsets parameter required by mlxtend >= 1.0
# Without it, newer versions raise: TypeError: association_rules() missing argument 'num_itemsets'
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0,
    num_itemsets=len(frequent_itemsets)
)
print(f"Total rules generated: {len(rules)}")
print(rules.head())

Total rules generated: 85174
                         antecedents                        consequents  \
0  (CHARLOTTE BAG DOLLY GIRL DESIGN)               ( DOLLY GIRL BEAKER)   
1               ( DOLLY GIRL BEAKER)  (CHARLOTTE BAG DOLLY GIRL DESIGN)   
2               ( DOLLY GIRL BEAKER)        (DOLLY GIRL CHILDRENS BOWL)   
3        (DOLLY GIRL CHILDRENS BOWL)               ( DOLLY GIRL BEAKER)   
4         (DOLLY GIRL CHILDRENS CUP)               ( DOLLY GIRL BEAKER)   

   antecedent support  consequent support   support  confidence       lift  \
0            0.058957            0.020408  0.011338    0.192308   9.423077   
1            0.020408            0.058957  0.011338    0.555556   9.423077   
2            0.020408            0.040816  0.015873    0.777778  19.055556   
3            0.040816            0.020408  0.015873    0.388889  19.055556   
4            0.036281            0.020408  0.013605    0.375000  18.375000   

   representativity  leverage  conviction  zhangs_m

## C2c: Top 3 Association Rules (Sorted by Lift)

In [27]:
# Sort all rules by lift descending and take the top 3
top_rules = rules.sort_values(by='lift', ascending=False).head(3)

# FIX: display top_rules directly — original top_rules.head() defaulted to 5 rows
# FIX: labelled output block for C2c screenshot requirement
print("=" * 70)
print("TOP 3 ASSOCIATION RULES — Sorted by Lift (Descending)")
print("=" * 70)
for i, row in top_rules.iterrows():
    print(f"\nRule {top_rules.index.get_loc(i) + 1}:")
    print(f"  Antecedent : {set(row['antecedents'])}")
    print(f"  Consequent : {set(row['consequents'])}")
    print(f"  Support    : {row['support']:.4f} ({row['support']*100:.2f}%)")
    print(f"  Confidence : {row['confidence']:.4f}")
    print(f"  Lift       : {row['lift']:.4f}")
print("\n" + "=" * 70)

# Also display as DataFrame for notebook rendering
top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

TOP 3 ASSOCIATION RULES — Sorted by Lift (Descending)

Rule 1:
  Antecedent : {'SPACEBOY BIRTHDAY CARD', 'CHILDRENS CUTLERY DOLLY GIRL ', 'ALARM CLOCK BAKELIKE PINK'}
  Consequent : {'ROUND SNACK BOXES SET OF4 WOODLAND ', 'ALARM CLOCK BAKELIKE RED ', 'CARD DOLLY GIRL '}
  Support    : 0.0113 (1.13%)
  Confidence : 1.0000
  Lift       : 88.2000

Rule 2:
  Antecedent : {'ROUND SNACK BOXES SET OF4 WOODLAND ', 'ALARM CLOCK BAKELIKE RED ', 'CHILDRENS CUTLERY DOLLY GIRL ', 'SPACEBOY BIRTHDAY CARD'}
  Consequent : {'ALARM CLOCK BAKELIKE PINK', 'CARD DOLLY GIRL '}
  Support    : 0.0113 (1.13%)
  Confidence : 1.0000
  Lift       : 88.2000

Rule 3:
  Antecedent : {'PLASTERS IN TIN SPACEBOY', 'ALARM CLOCK BAKELIKE RED ', 'SET6 RED SPOTTY PAPER PLATES', 'ALARM CLOCK BAKELIKE PINK'}
  Consequent : {'ROUND SNACK BOXES SET OF4 WOODLAND ', 'ALARM CLOCK BAKELIKE GREEN', 'PLASTERS IN TIN CIRCUS PARADE ', 'SET6 RED SPOTTY PAPER CUPS'}
  Support    : 0.0113 (1.13%)
  Confidence : 1.0000
  Lift       : 88.

,antecedents,consequents,support,confidence,lift
75149,"(SPACEBOY BIRTHDAY CARD, CHILDRENS CUTLERY DOL...","(ROUND SNACK BOXES SET OF4 WOODLAND , ALARM CL...",0.011338,1.0,88.2
75127,"(ROUND SNACK BOXES SET OF4 WOODLAND , ALARM CL...","(ALARM CLOCK BAKELIKE PINK, CARD DOLLY GIRL )",0.011338,1.0,88.2
84797,"(PLASTERS IN TIN SPACEBOY, ALARM CLOCK BAKELIK...","(ROUND SNACK BOXES SET OF4 WOODLAND , ALARM CL...",0.011338,1.0,88.2
